# Unidad 2: Aprendizaje Automático 
## 🎗️ Regresión Logística con Dataset Breast Cancer
### Inteligencia Artificial - Lic. en Sistemas - FCAD/UNER

![Hands](https://raw.githubusercontent.com/CristianPacifico/ia-ls-fcad-uner/main/notebooks/ml/images/pexels-tara-winstead-8385412.jpg)

[Foto de Tara Winstead](https://www.pexels.com/es-es/foto/persona-manos-sujetando-simbolo-8385412/)

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/ia-ls-fcad-uner/blob/main/notebooks/ml/05_LogisticRegression_Example.ipynb)

## Usando datasets integrados de scikit-learn

En este ejemplo usamos el dataset **Breast Cancer Wisconsin** incluido directamente en scikit-learn. Este dataset es un clásico para aprender clasificación binaria:

- 🔵 **Clase 1**: Tumor benigno
- 🔴 **Clase 0**: Tumor maligno

También exploraremos diferentes **configuraciones del modelo de Regresión Logística**, en particular el parámetro `max_iter` y el `solver` (algoritmo de optimización).

## 📦 1. Importación de librerías

scikit-learn incluye varios datasets listos para usar. Podés explorarlos en: https://scikit-learn.org/stable/datasets.html

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer  # Dataset integrado
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

## 📂 2. Carga y exploración del dataset

`load_breast_cancer()` devuelve un objeto tipo diccionario con los datos, nombres de features, y descripción. Lo convertimos en un DataFrame de pandas para facilitar su exploración.

In [ ]:
cancer_data = load_breast_cancer()

# Construir DataFrame con features + target
df = pd.DataFrame(cancer_data['data'], columns=cancer_data['feature_names'])
df['target'] = cancer_data['target']

# Mostrar con máximo 8 columnas para no desbordar la pantalla
pd.options.display.max_columns = 8

print("🔍 Dataset Breast Cancer (primeras filas):")
print(df.head())
print(f"\n📐 Estructura: {df.shape[0]} muestras x {df.shape[1]} columnas (incluyendo target)")
print(f"🏷️  Nombres de las clases: {list(cancer_data.target_names)}")

## 🔎 3. Distribución de clases

Un primer análisis importante: ¿Está el dataset balanceado?

In [ ]:
import matplotlib.pyplot as plt

conteo = df['target'].value_counts()
labels = [f"{cancer_data.target_names[i]} (clase {i})" for i in conteo.index]

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(labels, conteo.values, color=['tomato', 'steelblue'])
ax.set_title('📊 Distribución de clases — Breast Cancer')
ax.set_ylabel('Cantidad de muestras')
for i, v in enumerate(conteo.values):
    ax.text(i, v + 5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## ✂️ 4. Preparación de X e y, y separación de datos

In [ ]:
X = df[cancer_data.feature_names].values  # 30 features numéricas
y = df['target'].values                   # 0 = maligno, 1 = benigno

# 70% entrenamiento, 30% testeo
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=126
)

print(f"🏋️  Entrenamiento: {X_train.shape[0]} muestras, {X_train.shape[1]} features")
print(f"🧪  Testeo:        {X_test.shape[0]} muestras, {X_test.shape[1]} features")

## ⚙️ 5. Configuración del modelo: el parámetro `max_iter`

La Regresión Logística se ajusta de forma iterativa (minimiza una función de pérdida). Si el algoritmo no converge dentro del límite de iteraciones, lanza un **ConvergenceWarning**.

- El valor por defecto es `max_iter=100` — puede ser insuficiente para datasets con muchas features.
- Aumentarlo (ej: 5000) garantiza convergencia a costo de mayor tiempo de cómputo.

In [ ]:
model = LogisticRegression(verbose=1)

# Opción con más iteraciones para garantizar convergencia
# model = LogisticRegression(max_iter=5000, verbose=1)

# Alternativa cambiando el solver:
model = LogisticRegression(solver='liblinear', max_iter=5000, verbose=1)

model.fit(X_train, y_train)

print("✅ Modelo entrenado")
print(f"\n🔢 Solver usado: {model.solver}")
print(f"🔄 Iteraciones hasta convergencia: {model.n_iter_[0]}")

## 📐 6. Coeficientes del modelo

Los **coeficientes** (pesos) de la función sigmoide indican qué tanto contribuye cada feature a la predicción. Un coeficiente positivo aumenta la probabilidad de clase 1 (benigno), y uno negativo la disminuye.

In [ ]:
import numpy as np

coef_df = pd.DataFrame({
    'Feature': cancer_data.feature_names,
    'Coeficiente': model.coef_[0]
}).sort_values('Coeficiente', key=abs, ascending=False)

print("📊 Top 10 features más influyentes (por valor absoluto):")
print(coef_df.head(10).to_string(index=False))

print(f"\n⚓ Intercepto (bias): {model.intercept_[0]:.4f}")

## 🔮 7. Predicciones y evaluación

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)  # Probabilidades para cada clase

print("🎯 Evaluación del modelo:")
print(f"   Accuracy: {model.score(X_test, y_test):.4f}")
print()
print("📋 Reporte completo:")
from sklearn import metrics
print(metrics.classification_report(y_test, y_pred, target_names=cancer_data.target_names))

## 🔥 8. Matriz de confusión

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=cancer_data.target_names,
    colorbar=False,
    ax=ax
)
ax.set_title('🔥 Matriz de Confusión — Breast Cancer')
plt.tight_layout()
plt.show()

## 🏁 9. Conclusiones

- 🎗️ La Regresión Logística obtiene excelente performance en el dataset Breast Cancer.
- ⚙️ Usar `max_iter=5000` (o cambiar el `solver`) evita el `ConvergenceWarning`.
- 🔢 Los coeficientes del modelo reflejan el impacto de cada feature sobre la probabilidad de que el tumor sea benigno.
- 📦 Los datasets integrados de scikit-learn son perfectos para practicar sin necesidad de descargar archivos externos.
- 🔍 La función `predict_proba()` devuelve la **probabilidad** de cada clase, no solo la clase ganadora.